# NLDAS-2 MOSAIC — Soil Moisture

Visualize the consolidated NLDAS-2 MOSAIC land surface model soil moisture
from NASA GES DISC (NLDAS_MOS0125_M v2.0).

- **Variables:** `SoilM_0_10cm`, `SoilM_10_40cm`, `SoilM_40_200cm` (kg/m²)
- **Resolution:** 0.125° CONUS grid
- **Period:** 1979–present (monthly)
- **Note:** MOSAIC uses three soil layers; NOAH uses four (different layer boundaries)

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import xarray as xr

# --- Configuration ---
DATASTORE = Path("/caldera/hovenweep/projects/usgs/water/impd/nhgf/nhf-datastore")
PROJECT = Path("/caldera/hovenweep/projects/usgs/water/impd/nhgf/gfv2-spatial-targets")
NC_PATH = DATASTORE / "nldas_mosaic" / "nldas_mosaic_consolidated.nc"
PRIMARY_VAR = "SoilM_0_10cm"


In [ ]:
ds = xr.open_dataset(NC_PATH)
print(ds)
print(f"\nTime range: {ds.time.values[0]} to {ds.time.values[-1]}")
print(f"Time steps: {ds.time.size}")
print(f"Grid: {ds.lat.size} lat x {ds.lon.size} lon")
print(f"Data variables: {list(ds.data_vars)}")
print(f"Conventions: {ds.attrs.get('Conventions', 'N/A')}")
print(f"Grid mapping: {ds[PRIMARY_VAR].attrs.get('grid_mapping', 'N/A')}")


## Single-month soil moisture map

In [ ]:
TARGET_TIME = "2000-01-15"

da = ds[PRIMARY_VAR].sel(time=TARGET_TIME, method="nearest")
actual_time = str(da.time.values)[:10]

fig, ax = plt.subplots(figsize=(14, 8))
da.plot(ax=ax, cmap="YlGnBu", robust=True, cbar_kwargs={"label": "kg/m²"})
ax.set_title(f"NLDAS-2 MOSAIC Soil Moisture (0–10 cm) — {actual_time}")
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
ax.set_aspect("equal")
plt.tight_layout()
plt.show()

print(f"Stats for {actual_time}:")
print(f"  min:  {float(da.min()):.2f}")
print(f"  max:  {float(da.max()):.2f}")
print(f"  mean: {float(da.mean()):.2f}")
print(f"  NaN%: {float(da.isnull().mean()) * 100:.1f}%")


## All soil layers — single timestep comparison

In [ ]:
layer_vars = {
    "SoilM_0_10cm": "0–10 cm",
    "SoilM_10_40cm": "10–40 cm",
    "SoilM_40_200cm": "40–200 cm",
}

fig, axes = plt.subplots(1, 3, figsize=(20, 6))

for ax, (var, label) in zip(axes, layer_vars.items()):
    if var not in ds.data_vars:
        ax.set_visible(False)
        continue
    da = ds[var].sel(time=TARGET_TIME, method="nearest")
    da.plot(ax=ax, cmap="YlGnBu", robust=True, cbar_kwargs={"label": "kg/m²"})
    ax.set_title(f"{var}\n{label}")
    ax.set_aspect("equal")

fig.suptitle(f"NLDAS-2 MOSAIC Soil Moisture Layers — {actual_time}", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()


## Calibration period mean (2000–2009)

In [ ]:
conus = ds[PRIMARY_VAR].sel(
    time=slice("2000-01-01", "2009-12-31"),
)

mean_sm = conus.mean(dim="time")

fig, ax = plt.subplots(figsize=(14, 8))
mean_sm.plot(ax=ax, cmap="YlGnBu", robust=True, cbar_kwargs={"label": "kg/m²"})
ax.set_title("Mean Soil Moisture (0–10 cm) — NLDAS MOSAIC 2000–2009")
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
ax.set_aspect("equal")
plt.tight_layout()
plt.show()

print(f"2000-2009 mean stats:")
print(f"  min:  {float(mean_sm.min()):.2f}")
print(f"  max:  {float(mean_sm.max()):.2f}")
print(f"  mean: {float(mean_sm.mean()):.2f}")


## Seasonal comparison (2000–2009)

In [ ]:
seasons = {
    "DJF (Winter)": conus.sel(time=conus.time.dt.month.isin([12, 1, 2])),
    "MAM (Spring)": conus.sel(time=conus.time.dt.month.isin([3, 4, 5])),
    "JJA (Summer)": conus.sel(time=conus.time.dt.month.isin([6, 7, 8])),
    "SON (Fall)": conus.sel(time=conus.time.dt.month.isin([9, 10, 11])),
}

fig, axes = plt.subplots(2, 2, figsize=(16, 12))

for ax, (label, seasonal_data) in zip(axes.flatten(), seasons.items()):
    seasonal_mean = seasonal_data.mean(dim="time")
    seasonal_mean.plot(ax=ax, cmap="YlGnBu", robust=True, cbar_kwargs={"label": "kg/m²"})
    ax.set_title(f"{label} mean")
    ax.set_aspect("equal")

fig.suptitle("Seasonal Mean Soil Moisture (0–10 cm) — NLDAS MOSAIC 2000–2009", fontsize=14, y=1.01)
plt.tight_layout()
plt.show()


## Monthly time series (spatial mean)

In [ ]:
monthly_mean = ds[PRIMARY_VAR].mean(dim=["lat", "lon"])

fig, ax = plt.subplots(figsize=(16, 4))
monthly_mean.plot(ax=ax, color="#2980b9", linewidth=0.5)
ax.set_ylabel("Mean Soil Moisture (kg/m²)")
ax.set_title("NLDAS-2 MOSAIC spatial-mean monthly soil moisture (0–10 cm)")
ax.axvspan("2000-01-01", "2009-12-31", alpha=0.15, color="orange", label="Calibration period")
ax.legend()
plt.tight_layout()
plt.show()


## Monthly climatology (2000–2009)

In [ ]:
monthly_clim = conus.groupby("time.month").mean(dim=["time", "lat", "lon"])

fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(monthly_clim.month, monthly_clim.values, color="#2980b9", edgecolor="white")
ax.set_xlabel("Month")
ax.set_ylabel("Mean Soil Moisture (kg/m²)")
ax.set_title("Monthly Climatology — NLDAS MOSAIC 2000–2009 (0–10 cm)")
ax.set_xticks(range(1, 13))
ax.set_xticklabels(["J", "F", "M", "A", "M", "J", "J", "A", "S", "O", "N", "D"])
plt.tight_layout()
plt.show()


## CF compliance verification

In [ ]:
print("CF Compliance Checks:")
print(f"  Conventions attr:  {ds.attrs.get('Conventions', 'MISSING')}")
print(f"  Time dtype:        {ds.time.dtype}")
print(f"  Time first:        {ds.time.values[0]}")
print(f"  Time last:         {ds.time.values[-1]}")
print(f"  CRS variable:      {'crs' in ds.data_vars}")
if "crs" in ds.data_vars:
    print(f"  Grid mapping name: {ds['crs'].attrs.get('grid_mapping_name', 'MISSING')}")
    print(f"  CRS WKT:           {ds['crs'].attrs.get('crs_wkt', 'MISSING')[:60]}...")
print(f"  time_bnds:         {'time_bnds' in ds.data_vars}")
for var in ["SoilM_0_10cm", "SoilM_10_40cm", "SoilM_40_200cm"]:
    if var in ds.data_vars:
        print(f"  {var}:")
        print(f"    grid_mapping: {ds[var].attrs.get('grid_mapping', 'MISSING')}")
        print(f"    units:        {ds[var].attrs.get('units', 'MISSING')}")
        print(f"    long_name:    {ds[var].attrs.get('long_name', 'MISSING')}")


## Manifest provenance check

In [ ]:
import json

manifest_path = PROJECT / "manifest.json"
if manifest_path.exists():
    manifest = json.loads(manifest_path.read_text())
    entry = manifest["sources"].get("nldas_mosaic", {})
    print(f"Source key:       {entry.get('source_key', 'N/A')}")
    print(f"Access URL:       {entry.get('access_url', 'N/A')}")
    print(f"Period:           {entry.get('period', 'N/A')}")
    print(f"Variables:        {entry.get('variables', 'N/A')}")
    print(f"Consolidated NC:  {entry.get('consolidated_nc', 'N/A')}")
    print(f"Files downloaded: {len(entry.get('files', []))}")
    print(f"Downloaded UTC:   {entry.get('download_timestamp', 'N/A')}")
else:
    print(f"manifest.json not found at {manifest_path}")


In [ ]:
ds.close()
